# Informe de curvas MOP/OGLE + observaciones JS/HSH

Notebook reproducible para preparar la reunión: combina las curvas MOP/OGLE con las fechas de observación de JS y selecciona los sistemas observados con HSH cuya magnitud actual es mayor que el corte configurado.

El notebook no inventa observaciones: si falta el inventario JS, genera un PDF de estado y deja el flujo listo para continuar cuando se agregue el archivo.

In [1]:
from pathlib import Path
import sys
import json
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'monitoring_report.py').exists()), Path('/home/karennowo/target_selection'))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Parámetros editables
JS_DATA = REPO_ROOT / 'data' / 'js_data' / 'js_inventory.csv'
HSH_IMAGE_CATALOG = REPO_ROOT / 'data' / 'hsh_data' / 'image_collection_astro.csv'
HSH_TARGET_CATALOG = REPO_ROOT / 'data' / 'hsh_data' / 'hsh_objects.csv'
MOP_PHOTOMETRY_DIR = REPO_ROOT / 'notebooks' / 'outputs' / 'microlensing_followup' / 'cache' / 'mop_photometry'
CANDIDATE_TABLE = REPO_ROOT / 'notebooks' / 'outputs' / 'microlensing_followup' / 'all_candidate_events.csv'
OUTPUT_DIR = REPO_ROOT / 'notebooks' / 'outputs' / 'reunion_curvas_js_hsh'
HSH_MAG_LIMIT = 16.0
PLOTS_PER_PAGE = 3
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('REPO_ROOT:', REPO_ROOT)
print('JS_DATA:', JS_DATA, 'exists=', JS_DATA.exists())
print('MOP curves:', len(list(MOP_PHOTOMETRY_DIR.glob('*.csv'))))

REPO_ROOT: /home/karennowo/target_selection
JS_DATA: /home/karennowo/target_selection/data/js_data/js_inventory.csv exists= True
MOP curves: 67


In [2]:
from observatory_observations import load_hsh_astrometry_catalog, canonical_target_name
from monitoring_report import create_monitoring_report

def normalize_target_column(frame):
    frame = frame.copy()
    if 'Target' not in frame.columns:
        for col in ('target_id', 'target', 'object', 'source_target'):
            if col in frame.columns:
                frame['Target'] = frame[col]
                break
    if 'Target' not in frame.columns:
        raise ValueError('No se encontró una columna de identificación de sistema')
    frame['Target'] = frame['Target'].astype(str).str.strip()
    frame['target_key'] = frame['Target'].map(canonical_target_name)
    return frame

# Inventario astrométrico HSH: sólo imágenes object + astromet=yes.
hsh_epochs = load_hsh_astrometry_catalog(HSH_IMAGE_CATALOG)
hsh_epochs = hsh_epochs.rename(columns={'source_target': 'Target'})
hsh_epochs['provider'] = 'HSH'
hsh_epochs['target_key'] = hsh_epochs['Target'].map(canonical_target_name)
hsh_epochs = hsh_epochs[['Target', 'target_key', 'provider', 'mjd', 'exptime_s', 'band', 'usable']].copy()

# JS acepta nombres razonables de columnas y los normaliza al contrato del reporte.
if JS_DATA.exists():
    js_raw = pd.read_csv(JS_DATA)
    js_epochs = normalize_target_column(js_raw)
    if 'mjd' not in js_epochs.columns:
        for col in ('MJD', 'mjd_obs', 'mjd-obs'):
            if col in js_epochs.columns:
                js_epochs['mjd'] = js_epochs[col]
                break
    if 'mjd' not in js_epochs.columns and 'Timestamp' in js_epochs.columns:
        js_epochs['mjd'] = (pd.to_datetime(js_epochs['Timestamp'], errors='coerce') - pd.Timestamp('1858-11-17')) / pd.Timedelta(days=1)
    if 'mjd' not in js_epochs.columns and 'date' in js_epochs.columns:
        js_epochs['mjd'] = (pd.to_datetime(js_epochs['date'], errors='coerce') - pd.Timestamp('1858-11-17')) / pd.Timedelta(days=1)
    js_epochs['provider'] = 'JS'
    js_epochs['mjd'] = pd.to_numeric(js_epochs['mjd'], errors='coerce')
    js_epochs['exptime_s'] = pd.to_numeric(js_epochs.get('exptime_s', np.nan), errors='coerce')
    js_epochs['band'] = js_epochs.get('band', js_epochs.get('filter_or_grating', ''))
    js_epochs = js_epochs.dropna(subset=['mjd'])[['Target', 'target_key', 'provider', 'mjd', 'exptime_s', 'band']].copy()
else:
    js_raw = pd.DataFrame()
    js_epochs = pd.DataFrame(columns=['Target', 'target_key', 'provider', 'mjd', 'exptime_s', 'band'])

print(f'HSH epochs: {len(hsh_epochs):,}; HSH targets: {hsh_epochs.Target.nunique():,}')
print(f'JS epochs: {len(js_epochs):,}; JS targets: {js_epochs.Target.nunique():,}')

HSH epochs: 18,873; HSH targets: 35
JS epochs: 32; JS targets: 14


In [3]:
# Base de targets: parámetros MOP disponibles en la corrida previa.
if CANDIDATE_TABLE.exists():
    candidates = pd.read_csv(CANDIDATE_TABLE, low_memory=False).rename(columns={'target_id': 'Target'})
else:
    candidates = pd.read_csv(HSH_TARGET_CATALOG).rename(columns={'object': 'Target'})
candidates = normalize_target_column(candidates)

# Unimos los sistemas HSH con sus parámetros MOP y coordenadas.
hsh_targets = pd.read_csv(HSH_TARGET_CATALOG).rename(columns={'object': 'Target'})
hsh_targets = normalize_target_column(hsh_targets)
hsh_targets = hsh_targets.merge(candidates.drop_duplicates('target_key'), on='target_key', how='left', suffixes=('', '_candidate'))
hsh_targets['Target'] = hsh_targets['Target'].fillna(hsh_targets.get('Target_candidate'))

# Preferimos la magnitud actual MOP; si no existe, usamos magnitude/mag_now.
magnitude_candidates = [c for c in ('mop_mag_now', 'mag_now', 'magnitude') if c in hsh_targets.columns]
hsh_targets['magnitude_for_cut'] = np.nan
for col in magnitude_candidates:
    values = pd.to_numeric(hsh_targets[col], errors='coerce')
    values = values.where(values.gt(0))
    hsh_targets['magnitude_for_cut'] = hsh_targets['magnitude_for_cut'].fillna(values)

weak_hsh = hsh_targets.loc[hsh_targets['magnitude_for_cut'].gt(HSH_MAG_LIMIT)].copy()
weak_hsh = weak_hsh[['Target', 'target_key', 'ra_deg', 'dec_deg', 'magnitude_for_cut'] + [c for c in ('mop_t_0_hjd', 'mop_t_e_days', 'mop_mag_now', 'mag_now', 'magnitude') if c in hsh_targets.columns]]
weak_hsh = weak_hsh.drop_duplicates('Target').reset_index(drop=True)

# La curva JS requiere sistemas JS; si no hay catálogo, queda explícitamente vacío.
js_names = js_epochs[['Target', 'target_key']].drop_duplicates().reset_index(drop=True)
js_targets = js_names.merge(candidates.drop_duplicates('target_key'), on='target_key', how='left', suffixes=('', '_candidate'))

print(f'HSH débiles (mag > {HSH_MAG_LIMIT:g}): {len(weak_hsh)}')
print(f'Sistemas JS con curva candidata: {len(js_targets)}')
display(weak_hsh[['Target', 'magnitude_for_cut']].sort_values('magnitude_for_cut').head(20))

HSH débiles (mag > 16): 20
Sistemas JS con curva candidata: 14


,Target,magnitude_for_cut
1,OGLE-2024-BLG-0034,16.152960
3,OGLE-2025-BLG-1457,16.209984
6,OGLE-2024-BLG-0010,16.229306
15,OGLE-2024-GD-0001,16.293859
0,OGLE-2025-GD-0001,16.304571
11,Gaia24ddl,16.465462
16,OGLE-2024-BLG-0009,16.728055
7,Gaia24bsj,16.866290
19,OGLE-2024-GD-0006,16.919695
12,OGLE-2024-DG-0006,16.919695


In [4]:
# Resumen auditable y PDFs. Las épocas HSH se incluyen sólo para los HSH débiles.
weak_hsh_epochs = hsh_epochs.loc[hsh_epochs['target_key'].isin(weak_hsh['target_key'])].copy()
js_epochs_selected = js_epochs.loc[js_epochs['target_key'].isin(js_targets['target_key'])].copy()

summary_rows = []
for _, row in pd.concat([js_targets.assign(section='JS'), weak_hsh.assign(section='HSH mag>16')], ignore_index=True).iterrows():
    key = row['target_key']
    summary_rows.append({
        'Target': row['Target'],
        'section': row['section'],
        'magnitude_for_cut': row.get('magnitude_for_cut', np.nan),
        'mop_curve_exists': (MOP_PHOTOMETRY_DIR / f"{row['Target']}.csv").exists(),
        'js_epochs': int((js_epochs['target_key'] == key).sum()),
        'hsh_epochs': int((hsh_epochs['target_key'] == key).sum()),
    })
summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / 'summary.csv'
summary.to_csv(summary_path, index=False)

# Informe para HSH débiles: curvas MOP/OGLE más fechas HSH.
hsh_pdf = OUTPUT_DIR / 'hsh_debiles_mag_mayor_16.pdf'
if len(weak_hsh):
    create_monitoring_report(weak_hsh, hsh_pdf, mop=None, mop_photometry_dir=MOP_PHOTOMETRY_DIR, observatory_epochs=weak_hsh_epochs, layers=('mop_photometry', 'hsh'), plots_per_page=PLOTS_PER_PAGE)

# Informe JS: sólo se crea con observaciones reales disponibles.
js_pdf = OUTPUT_DIR / 'curvas_mop_ogle_mas_fechas_js.pdf'
if len(js_targets):
    create_monitoring_report(js_targets, js_pdf, mop=None, mop_photometry_dir=MOP_PHOTOMETRY_DIR, observatory_epochs=js_epochs_selected, layers=('mop_photometry', 'js'), plots_per_page=PLOTS_PER_PAGE)

# PDF único de estado para compartir en la reunión.
meeting_pdf = OUTPUT_DIR / 'informe_reunion_curvas_js_hsh.pdf'
with PdfPages(meeting_pdf) as pdf:
    fig = plt.figure(figsize=(11.7, 8.3))
    fig.text(0.08, 0.88, 'Informe de curvas para reunión', fontsize=22, weight='bold')
    fig.text(0.08, 0.78, f'Curvas MOP/OGLE + fechas JS: {len(js_targets)} sistemas', fontsize=14)
    fig.text(0.08, 0.71, f'Sistemas HSH con magnitud > {HSH_MAG_LIMIT:g}: {len(weak_hsh)}', fontsize=14)
    status = 'DISPONIBLE' if JS_DATA.exists() else 'PENDIENTE: no se encontró data/js_data/js_inventory.csv'
    fig.text(0.08, 0.62, f'Inventario JS: {status}', fontsize=14, color='darkgreen' if JS_DATA.exists() else 'darkred')
    fig.text(0.08, 0.48, 'Fuentes:', fontsize=13, weight='bold')
    fig.text(0.11, 0.42, f'- MOP/OGLE: {MOP_PHOTOMETRY_DIR}', fontsize=10)
    fig.text(0.11, 0.37, f'- HSH: {HSH_IMAGE_CATALOG}', fontsize=10)
    fig.text(0.11, 0.32, f'- JS: {JS_DATA}', fontsize=10)
    fig.text(0.08, 0.18, 'El informe no interpreta la ausencia de JS como ausencia de observación; sólo indica que el inventario no está disponible localmente.', fontsize=10, wrap=True)
    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

# Unir portada + sección JS + sección HSH en el PDF principal.
if js_pdf.exists() and hsh_pdf.exists():
    combined_pdf = OUTPUT_DIR / 'informe_reunion_curvas_js_hsh_completo.pdf'
    subprocess.run(['gs', '-q', '-dBATCH', '-dNOPAUSE', '-sDEVICE=pdfwrite', f'-sOutputFile={combined_pdf}', str(meeting_pdf), str(js_pdf), str(hsh_pdf)], check=True)
    combined_pdf.replace(meeting_pdf)

print('Archivos generados:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f' - {path.name} ({path.stat().st_size:,} bytes)')

Photometry skipped for OGLE-2024-GD-0006: no MOP client or cached photometry


Photometry skipped for OGLE-2024-GD-0006: no MOP client or cached photometry


Photometry skipped for ZTF25aakjuzb: no MOP client or cached photometry
Photometry skipped for ZTF25acafija: no MOP client or cached photometry
Photometry skipped for ZTF25acfsvov: no MOP client or cached photometry


Photometry skipped for sn2026ngr: no MOP client or cached photometry


Archivos generados:
 - curvas_mop_ogle_mas_fechas_js.pdf (101,874 bytes)
 - hsh_debiles_mag_mayor_16.pdf (195,801 bytes)
 - informe_reunion_curvas_js_hsh.pdf (1,160,906 bytes)
 - summary.csv (1,624 bytes)
